# jev-benchmark

面向语义决策模型的中文言下之意评测集。本 notebook 依次：加载本地模型并演示三种决策原语 → 运行评测 → 渲染报表。安装与模型说明见 [README](README.md)。

## 1. 本地模型与三种决策原语

`noul`（是/否）、`choice`（多选一）、`score`（有序评分）。本地模型读取首个生成位置的选项字母 logits 并仅在候选项间归一化，概率未经校准。

In [ ]:
from jev_benchmark.local_model import LocalModel

local = LocalModel()

urgency = local.score("救命！我的付款已经连续 3 天失败了。我今天必须拿到这笔钱。", "这条消息是否表达了紧迫性？",
                      {"A": "否", "B": "是"})
team = local.score("同一笔订单向我收了两次款。请退还重复支付的金额。", "哪个团队应该处理这项客户请求？",
                   {"A": "账单团队 — 付款与退款", "B": "配送团队 — 配送问题", "C": "技术团队 — 产品故障"})
severity = local.score("这个应用今天已经崩溃五次了。我无法完成工作，感到非常沮丧。", "客户的沮丧程度有多严重？",
                       {"A": "0 — 无", "B": "1 — 轻微", "C": "2 — 中等", "D": "3 — 严重"})

print("noul   紧迫性（A=否 / B=是）：", {k: round(v, 3) for k, v in urgency.items()})
print("choice 处理团队：", {k: round(v, 3) for k, v in team.items()})
print(f"score  沮丧程度期望值（0–3）：{sum(i * p for i, p in enumerate(severity.values())):.2f}")

## 2. 运行评测

默认只评测本地模型，不读取 `.env`、不联网。在 `MODELS` 中追加 `"jev"`（官方 API）或 `"intranet"`（内网模型）即可同题对比；**每个远程模型全量运行发起 100 次请求，官方 API 可能计费**。结果写入 `results/<模型组合>.json`。

In [ ]:
from jev_benchmark import DATA_PATH
from jev_benchmark.evaluate import load_contestant, result_path, run_benchmark

MODELS = ["local"]

contestants = [local if key == "local" else load_contestant(key) for key in MODELS]
output_path = result_path(MODELS)
report = run_benchmark(DATA_PATH, contestants, output_path,
                       lambda done, total: print(f"\r已完成 {done}/{total}", end=""))
print(f"\n已保存：{output_path.name}")

## 3. 查看报表

只读取已保存的结果，不调用模型；结果与当前题集哈希不一致时拒绝展示。

In [ ]:
import hashlib
import json
from IPython.display import HTML, display
from jev_benchmark import DATA_PATH, RESULTS_DIR
from jev_benchmark.evaluate import render_report

REPORT = "local.json"

raw = DATA_PATH.read_bytes()
saved = json.loads((RESULTS_DIR / REPORT).read_text(encoding="utf-8"))
if saved["dataset_sha256"] != hashlib.sha256(raw).hexdigest():
    raise ValueError("结果与当前题集不一致，请重新运行评测。")
display(HTML(render_report(saved, json.loads(raw))))